# Agent with Memory

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
model = ChatGroq(model="qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020FB5741190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020FB68706D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

---
**✅ InMemorySaver in LangGraph**

`langgraph.checkpoint.memory` is a module in **LangGraph** that provides tools for saving the state of a workflow in memory.

`InMemorySaver` is a class from this module that stores **checkpoints** (the current state of an agent or workflow) temporarily in the computer’s **RAM**.

**Idea:**  
When an AI agent built with LangGraph runs, it executes multiple steps. A **checkpoint** saves the agent’s current state so it can continue from that point later.

`InMemorySaver` keeps these checkpoints **only in memory**, meaning the data is lost when the program stops.

---

In [12]:
# Agent with the Property InMemorySaver
agent = create_agent(
    model,
    checkpointer=InMemorySaver()
)

In [13]:
# Run with Thread id
config = {"configurable":{"thread_id":"test1"}}

---
`# Thread ID Configuration :`

**Code:**

config = {"configurable": {"thread_id": "test1"}}

### Meaning

- `thread_id` is used to **identify a conversation session**.
- It allows the model to **maintain memory across multiple calls**.
- All requests with the same **thread_id** belong to the **same conversation thread**.


`# Why Thread ID is Used`

- Helps **store and retrieve conversation history**.
- Enables **persistent memory in agents or workflows**.
- Allows multiple users to interact with the system **independently**.



`# Example Usage`

response = agent.invoke(
    {"messages":[{"role":"user","content":"Hello"}]},
    config={"configurable":{"thread_id":"test1"}}
)

Later we can continue the same conversation:

response = agent.invoke(
    {"messages":[{"role":"user","content":"What did I ask before?"}]},
    config={"configurable":{"thread_id":"test1"}}
)

The agent will **remember previous messages**.



`# Key Idea`

thread_id → **conversation identifier**

Different thread_ids create **different independent conversations**.

Example:

thread_id = "user1_chat" → conversation 1  
thread_id = "user2_chat" → conversation 2

---

In [14]:
from langchain_core.messages import HumanMessage,SystemMessage
response = agent.invoke({"messages":[HumanMessage(content="Hi!! My name is Paras")]},config=config)
response

{'messages': [HumanMessage(content='Hi!! My name is Paras', additional_kwargs={}, response_metadata={}, id='57ce1001-6c2c-4c8f-b6e9-e50bef5cf3af'),
  AIMessage(content='<think>\nOkay, so the user said Hi and introduced themselves as Paras. I need to respond appropriately. First, I should acknowledge their greeting and offer help. Maybe say something like "Hi, Paras! How can I assist you today?" Keep it friendly and open-ended. No need for markdown, just plain text. Let them know I\'m here to help with any questions or tasks. Make sure the tone is welcoming and approachable.\n</think>\n\nHi, Paras! How can I assist you today? 😊 I\'m here to help with any questions, tasks, or just friendly chat. What\'s on your mind?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 14, 'total_tokens': 140, 'completion_time': 0.26182856, 'completion_tokens_details': None, 'prompt_time': 0.000353543, 'prompt_tokens_details': None, 'queue_time': 0.1604249

In [16]:
agent.invoke({"messages":[HumanMessage(content="Hi!! What is my name?")]},config=config)
# since we use same config, it will remember the previous conversation
# so it will remember the name Paras
# so it will respond with "Hi again, Paras!

{'messages': [HumanMessage(content='Hi!! My name is Paras', additional_kwargs={}, response_metadata={}, id='57ce1001-6c2c-4c8f-b6e9-e50bef5cf3af'),
  AIMessage(content='<think>\nOkay, so the user said Hi and introduced themselves as Paras. I need to respond appropriately. First, I should acknowledge their greeting and offer help. Maybe say something like "Hi, Paras! How can I assist you today?" Keep it friendly and open-ended. No need for markdown, just plain text. Let them know I\'m here to help with any questions or tasks. Make sure the tone is welcoming and approachable.\n</think>\n\nHi, Paras! How can I assist you today? 😊 I\'m here to help with any questions, tasks, or just friendly chat. What\'s on your mind?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 14, 'total_tokens': 140, 'completion_time': 0.26182856, 'completion_tokens_details': None, 'prompt_time': 0.000353543, 'prompt_tokens_details': None, 'queue_time': 0.1604249

In [17]:
# Another thread id for another user/conversation
config_2 = {"configurable":{"thread_id":"test2"}}
from langchain_core.messages import HumanMessage,SystemMessage
response = agent.invoke({"messages":[HumanMessage(content="Hi!! My name is Riyan")]},config=config_2)
response

{'messages': [HumanMessage(content='Hi!! My name is Riyan', additional_kwargs={}, response_metadata={}, id='bf6a59aa-0ce2-4ba8-8d8f-d521ac7dc6e0'),
  AIMessage(content='<think>\nOkay, the user introduced themselves as Riyan. I should respond in a friendly and welcoming manner. Let me make sure to acknowledge their name and offer help. Maybe ask how I can assist them today. Keep it simple and positive.\n</think>\n\nHello, Riyan! Nice to meet you! 😊 How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 15, 'total_tokens': 86, 'completion_time': 0.128750607, 'completion_tokens_details': None, 'prompt_time': 0.000525762, 'prompt_tokens_details': None, 'queue_time': 0.045952948, 'total_time': 0.129276369}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cde2c-aed8-7ca0-b6ae-4b43e123

In [19]:
agent.invoke({"messages":[HumanMessage(content="Hi!! What is my name?")]},config=config_2)
# Since know we use config_2,it will remember the name Riyan which is related to config_2

{'messages': [HumanMessage(content='Hi!! My name is Riyan', additional_kwargs={}, response_metadata={}, id='bf6a59aa-0ce2-4ba8-8d8f-d521ac7dc6e0'),
  AIMessage(content='<think>\nOkay, the user introduced themselves as Riyan. I should respond in a friendly and welcoming manner. Let me make sure to acknowledge their name and offer help. Maybe ask how I can assist them today. Keep it simple and positive.\n</think>\n\nHello, Riyan! Nice to meet you! 😊 How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 15, 'total_tokens': 86, 'completion_time': 0.128750607, 'completion_tokens_details': None, 'prompt_time': 0.000525762, 'prompt_tokens_details': None, 'queue_time': 0.045952948, 'total_time': 0.129276369}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cde2c-aed8-7ca0-b6ae-4b43e123